# Блокнот 01: Профілювання якості даних та первинне очищення

**Фаза 1 нашого проєкту (Data Ingestion & Quality Gate 1)**

У цьому блокноті реалізовано:
1. Завантаження сирого датасету `data/raw/official_data_uk.csv`.
2. Фільтрація за рівнем `level == 'oblast'` для консистентності 2022–2026 років.
3. Конвертація часових поясів (UTC -> `Europe/Kyiv`).
4. Виявлення та фільтрація аномалій (мікро-глітчі < 60с, записи з ТОТ).
5. Хронологічний 3-way спліт (Train / Validation / Test).
6. Верифікація за критеріями **Quality Gate 1**.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

parquet_path = Path("../data/processed/alerts_clean_oblast.parquet")
df = pd.read_parquet(parquet_path)
print(f"Завантажено очищених рядків: {len(df):,}")
df.head()

Завантажено очищених рядків: 84,908


,oblast,raion,hromada,level,started_at,finished_at,source,is_finished,duration_seconds,duration_minutes,...,started_at_kyiv,finished_at_kyiv,start_year,start_month,start_date,start_hour_kyiv,start_day_of_week,start_day_num,is_night,split_set
0,Вінницька область,None,None,oblast,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,official,True,2373.0,39.550000,...,2022-03-15 18:10:34+02:00,2022-03-15 18:50:07+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
1,Житомирська область,None,None,oblast,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,official,True,2578.0,42.966667,...,2022-03-15 18:11:25+02:00,2022-03-15 18:54:23+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
2,Кіровоградська область,None,None,oblast,2022-03-15 16:15:11+00:00,2022-03-15 16:54:52+00:00,official,True,2381.0,39.683333,...,2022-03-15 18:15:11+02:00,2022-03-15 18:54:52+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
3,Івано-Франківська область,None,None,oblast,2022-03-15 16:17:28+00:00,2022-03-15 17:14:18+00:00,official,True,3410.0,56.833333,...,2022-03-15 18:17:28+02:00,2022-03-15 19:14:18+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
4,Львівська область,None,None,oblast,2022-03-15 16:17:29+00:00,2022-03-15 17:16:23+00:00,official,True,3534.0,58.900000,...,2022-03-15 18:17:29+02:00,2022-03-15 19:16:23+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train


## 1. Загальна інформація про датасет

In [2]:
print(f"Діапазон спостережень: {df['started_at_kyiv'].min()} — {df['started_at_kyiv'].max()}")
print(f"Кількість унікальних областей: {df['oblast'].nunique()}")
print("\nРозподіл за вибірками хронологічного спліту:")
print(df['split_set'].value_counts())
df.info()

Діапазон спостережень: 2022-03-15 18:10:34+02:00 — 2026-09-07 07:38:21+03:00
Кількість унікальних областей: 25

Розподіл за вибірками хронологічного спліту:
split_set
train    36933
test     26111
val      21864
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 84908 entries, 0 to 84907
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype                      
---  ------               --------------  -----                      
 0   oblast               84908 non-null  str                        
 1   raion                0 non-null      object                     
 2   hromada              0 non-null      object                     
 3   level                84908 non-null  str                        
 4   started_at           84908 non-null  datetime64[us, UTC]        
 5   finished_at          84908 non-null  datetime64[us, UTC]        
 6   source               84908 non-null  str                        
 7   is_finished          84908 no

## 2. Аналіз тривалості тривог (хвилини)

In [3]:
regular_df = df[~df["is_occupied_extreme"] & df["is_finished"]]

print("Статистика тривалості (хвилини):")
print(regular_df["duration_minutes"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

Статистика тривалості (хвилини):
count     84907.000000
mean        102.400861
std         841.316231
min           1.000000
25%          26.383333
50%          43.150000
75%         100.658333
90%         210.140000
95%         329.056667
99%         704.094000
max      201561.683333
Name: duration_minutes, dtype: float64


## 3. Топ областей за кількістю оголошених тривог

In [4]:
oblast_counts = regular_df["oblast"].value_counts()
print("Топ-5 найбільш активних областей:")
print(oblast_counts.head(5))

print("\nТоп-5 найбезпечніших областей (глибокий тил):")
print(oblast_counts.tail(5))

Топ-5 найбільш активних областей:
oblast
Харківська область    8780
Запорізька область    8185
Донецька область      7218
Сумська область       6055
Полтавська область    5862
Name: count, dtype: int64

Топ-5 найбезпечніших областей (глибокий тил):
oblast
Волинська область            933
Чернівецька область          868
Львівська область            848
Івано-Франківська область    839
Закарпатська область         753
Name: count, dtype: int64


## 4. Перевірка Quality Gate 1

In [5]:
assert len(df) > 70000, "Gate 1 Fail: Row count too low!"
assert (df[df["is_finished"]]["duration_seconds"] < 0).sum() == 0, "Gate 1 Fail: Negative durations!"
assert df["started_at_kyiv"].notna().all(), "Gate 1 Fail: Missing timezone conversions!"
print("🎉 QUALITY GATE 1 УСПІШНО ПРОЙДЕНО!")

🎉 QUALITY GATE 1 УСПІШНО ПРОЙДЕНО!
